# Notebook 01 — Baseline DQN on CartPole-v1

**APBIO Semester Project** — Universidad de Vigo, 2025/26

This notebook establishes the **plain DQN baseline** on `CartPole-v1`. It:

1. Mounts Google Drive (so results survive runtime resets)
2. Installs required packages
3. Sets deterministic seeds
4. Trains a standard DQN agent using Stable-Baselines3
5. Logs the learning curve and saves everything to Drive

Runtime: **about 3–5 minutes on CPU** (no GPU needed for CartPole).

## 1. Mount Google Drive

We store logs, checkpoints, and figures under `MyDrive/apbio_project/` so they persist across Colab sessions.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, pathlib

PROJECT_ROOT = pathlib.Path('/content/drive/MyDrive/apbio_project')
(PROJECT_ROOT / 'logs').mkdir(parents=True, exist_ok=True)
(PROJECT_ROOT / 'figures').mkdir(parents=True, exist_ok=True)
(PROJECT_ROOT / 'checkpoints').mkdir(parents=True, exist_ok=True)

os.chdir(PROJECT_ROOT)
print('Working directory:', os.getcwd())
print('Contents:', os.listdir('.'))

## 2. Install dependencies

Colab already ships with PyTorch and NumPy. We only need to add Stable-Baselines3 and Gymnasium.

In [ ]:
%pip install -q 'gymnasium>=0.29' 'stable-baselines3>=2.3' 'pyyaml>=6.0'

In [ ]:
# Quick import sanity check
import gymnasium as gym
import stable_baselines3 as sb3
import torch
import numpy as np

print('gymnasium     :', gym.__version__)
print('sb3           :', sb3.__version__)
print('torch         :', torch.__version__)
print('numpy         :', np.__version__)
print('cuda available:', torch.cuda.is_available())

## 3. Reproducibility helpers

The course specification requires **at least 5 independent seeds** for every reported result. We wrap the seeding logic in a single helper so every experiment is reproducible.

In [ ]:
import random
import numpy as np
import torch

def set_global_seed(seed: int) -> None:
    """Seed every RNG we know about so runs are reproducible."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_global_seed(0)
print('Seed set.')

## 4. The environment — CartPole-v1

`CartPole-v1`:

- **Observation:** 4 continuous features (cart position, cart velocity, pole angle, pole angular velocity)
- **Action:** 2 discrete actions (push cart left / right)
- **Reward:** +1 per step
- **Episode ends:** when the pole falls > 15° or the cart leaves the track; maximum return is **500**.

In [ ]:
env = gym.make('CartPole-v1')
obs, info = env.reset(seed=0)
print('Observation space:', env.observation_space)
print('Action space     :', env.action_space)
print('Sample observation:', obs)
env.close()

## 5. Training callback — learning curve logger

Stable-Baselines3 provides TensorBoard integration, but we want **per-episode CSV logs** to satisfy the course requirement (raw logs `.csv` per seed). We implement a tiny custom callback.

In [ ]:
import csv
from pathlib import Path
from stable_baselines3.common.callbacks import BaseCallback


class EpisodeLogger(BaseCallback):
    """Logs (episode, timestep, return, length) to a CSV file.

    SB3 stores finished episodes in `self.model.ep_info_buffer` (a deque of
    dicts). We drain it after every rollout and append new rows.
    """

    def __init__(self, csv_path: str, verbose: int = 0):
        super().__init__(verbose)
        self.csv_path = Path(csv_path)
        self.csv_path.parent.mkdir(parents=True, exist_ok=True)
        self._episode_count = 0
        self._seen = 0

        # Write header once.
        with self.csv_path.open('w', newline='') as f:
            writer = csv.writer(f)
            writer.writerow(['episode', 'timestep', 'return', 'length'])

    def _on_step(self) -> bool:
        # ep_info_buffer grows as episodes finish; we read only new entries.
        buf = list(self.model.ep_info_buffer)
        new_entries = buf[self._seen:]
        if new_entries:
            rows = []
            for ep in new_entries:
                self._episode_count += 1
                rows.append([
                    self._episode_count,
                    self.num_timesteps,
                    float(ep['r']),
                    int(ep['l']),
                ])
            with self.csv_path.open('a', newline='') as f:
                writer = csv.writer(f)
                writer.writerows(rows)
            self._seen = len(buf)
        return True


print('EpisodeLogger callback defined.')

## 6. Train the baseline DQN — short smoke test (1 seed, 30k steps)

This is **not** the full experiment — just a quick sanity check that the stack works and CartPole is solvable in our setup. The full 5-seed sweep comes in Notebook 04.

In [ ]:
import time
from stable_baselines3 import DQN

SEED = 0
TOTAL_TIMESTEPS = 30_000      # ~3 min on CPU; enough to see learning begin
LOG_PATH = PROJECT_ROOT / 'logs' / f'baseline_seed{SEED}.csv'

set_global_seed(SEED)
env = gym.make('CartPole-v1')

model = DQN(
    policy='MlpPolicy',
    env=env,
    learning_rate=5e-4,
    buffer_size=50_000,
    learning_starts=1_000,
    batch_size=64,
    tau=1.0,
    gamma=0.99,
    train_freq=4,
    gradient_steps=1,
    target_update_interval=500,
    exploration_fraction=0.2,
    exploration_initial_eps=1.0,
    exploration_final_eps=0.05,
    policy_kwargs=dict(net_arch=[64, 64]),
    seed=SEED,
    verbose=0,
)

callback = EpisodeLogger(csv_path=str(LOG_PATH))

t0 = time.time()
model.learn(total_timesteps=TOTAL_TIMESTEPS, callback=callback, progress_bar=True)
elapsed = time.time() - t0

print(f'\nTraining finished in {elapsed:.1f} s')
print(f'Log saved to: {LOG_PATH}')

## 7. Plot the learning curve

A 20-episode rolling mean is a standard smoothing choice for CartPole. If the agent is learning, we expect the return to rise from ~20 (random) toward 500 (maximum).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv(LOG_PATH)
df['return_smooth'] = df['return'].rolling(window=20, min_periods=1).mean()

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(df['episode'], df['return'], alpha=0.25, label='raw episode return')
ax.plot(df['episode'], df['return_smooth'], linewidth=2, label='20-ep rolling mean')
ax.axhline(500, color='grey', linestyle='--', linewidth=1, label='CartPole-v1 max (500)')
ax.set_xlabel('Episode')
ax.set_ylabel('Return')
ax.set_title(f'Baseline DQN — CartPole-v1 (seed {SEED}, {TOTAL_TIMESTEPS} steps)')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)
fig.tight_layout()

out_fig = PROJECT_ROOT / 'figures' / f'baseline_smoketest_seed{SEED}.png'
fig.savefig(out_fig, dpi=150)
plt.show()
print('Figure saved to:', out_fig)

## 8. Sanity checks

Before moving on, let's verify:

1. The CSV has the expected columns and a reasonable number of episodes.
2. The final mean return is notably higher than the initial mean return (i.e. the agent actually learned something).

In [ ]:
print('Rows in log         :', len(df))
print('Columns             :', list(df.columns))
print('First 10 ep mean    :', df['return'].head(10).mean())
print('Last 10 ep mean     :', df['return'].tail(10).mean())
print('Max return observed :', df['return'].max())

## 9. Next steps

If the final 10-episode mean is above ~150 and clearly higher than the first 10, our baseline stack works. Next notebook (`02_hopfield_module.ipynb`):

1. Implement the Modern Hopfield Network as a standalone PyTorch module.
2. Unit-test associative retrieval on synthetic patterns.
3. Wrap it in a `HopfieldReplayBuffer` compatible with SB3's DQN.